In [1]:
import os
import pandas as pd
import torch
import tqdm
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
from utils.datasetv2 import PoseDatasetV2
from collections import Counter
from typing import List, Dict, Tuple
import numpy as np
import time
from utils.utils import GaussianNoise, send_inputs_to_device, decode_predictions, decode_labels, load_and_process_text_data
from utils.metrics import wer_list
import random
import argparse
from torch.nn.utils.rnn import pad_sequence
from utils.utils import generate_autoregressive

import torch
import torch.nn.functional as F
from tqdm.auto import tqdm
import os
from utils.utils import decode_labels
from enc_dec_models.models import TrOCRMyDecoder
torch.set_float32_matmul_precision('high')
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
def custom_collate_fn(batch):
    """Custom collate function to handle variable-length sequences"""
    file_paths = [item['file_path'] for item in batch]
    pose_values = [item['pose_values'] for item in batch]
    input_ids = [item['input_ids'] for item in batch]
    attention_masks = [item['attention_mask'] for item in batch]
    labels = [item['labels'] for item in batch]
    
    pose_values_padded = pad_sequence(pose_values, batch_first=True, padding_value=0.0)
    input_ids_padded = pad_sequence(input_ids, batch_first=True, padding_value=0)
    attention_masks_padded = pad_sequence(attention_masks, batch_first=True, padding_value=0)
    labels_padded = pad_sequence(labels, batch_first=True, padding_value=-100)
    input_lens          = [len(item['input_ids'])-1  for item in batch]
    return {
        'file_path': file_paths,
        'pose_values': pose_values_padded,
        'input_ids': input_ids_padded,
        'attention_mask': attention_masks_padded,
        'labels': labels_padded,
        'input_lens': input_lens
    }


In [3]:
def setup_training_data():
    """Setup data loaders"""
    train_csv = "annotations_v2/SI/train.txt"
    dev_csv = "annotations_v2/SI/dev.txt"
    
    train_processed, dev_processed, vocab_map, inv_vocab_map, vocab_list = load_and_process_text_data(
        train_csv, dev_csv, target_column='gloss'
    )
    
    dataset_train = PoseDatasetV2(
        dataset_name2="isharah",
        label_csv=train_csv,
        split_type="train",
        target_enc_df=train_processed,
        augmentations=True,
        augmentation_config='aggressive',
        transform=transforms.Compose([GaussianNoise()])
    )
    
    dataset_dev = PoseDatasetV2(
        dataset_name2="isharah", 
        label_csv=dev_csv,
        split_type="dev",
        target_enc_df=dev_processed,
        augmentations=False
    )
    
    train_loader = DataLoader(
        dataset_train, 
        batch_size=32, 
        shuffle=True, 
        num_workers=4,
        collate_fn=custom_collate_fn 
    )
    
    dev_loader = DataLoader(
        dataset_dev, 
        batch_size=1, 
        shuffle=False, 
        num_workers=4,
        collate_fn=custom_collate_fn  
    )
    
    vocab_info = {
        'vocab_map': vocab_map, 
        'inv_vocab_map': inv_vocab_map, 
        'vocab_list': vocab_list,
        'vocab_size': len(vocab_list)
    }
    
    print(f"[DATA] Training: {len(dataset_train)}, Dev: {len(dataset_dev)}, Vocab: {len(vocab_list)}")
    return train_loader, dev_loader, vocab_info

def save_checkpoint(model, optimizer, epoch, train_loss, val_loss, val_wer, checkpoint_path):
    """Save model checkpoint"""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'train_loss': train_loss,
        'val_loss': val_loss,
        'val_wer': val_wer,
    }
    torch.save(checkpoint, checkpoint_path)
    print(f"Checkpoint saved: {checkpoint_path}")

def save_best_model(model, optimizer, epoch, train_loss, val_loss, val_wer, work_dir):
    """Save the best model based on WER"""
    best_model_path = os.path.join(work_dir, 'best_model.pt')
    save_checkpoint(model, optimizer, epoch, train_loss, val_loss, val_wer, best_model_path)
    print(f"NEW BEST MODEL SAVED! WER: {val_wer:.4f}")


In [4]:
train_loader, dev_loader, vocab_info = setup_training_data()

Loaded 10000 train, 949 dev samples
Created vocabulary with 684 tokens
Using augmentation config: aggressive
Loading pose data from data/pose_data_isharah1000_hands_lips_body_May12.pkl


Loaded 9500 samples for split: train
Using augmentation config: moderate
Loading pose data from data/pose_data_isharah1000_hands_lips_body_May12.pkl
Loaded 949 samples for split: dev
[DATA] Training: 9500, Dev: 949, Vocab: 684


In [5]:
# from tokenizers import Tokenizer
# from tokenizers.models import WordLevel
# from tokenizers.pre_tokenizers import Whitespace
# from transformers import PreTrainedTokenizerFast


# # Step 1: Create a tokenizer using WordLevel
# tokenizer = Tokenizer(WordLevel(vocab=vocab_info["vocab_map"], unk_token="<unk>"))
# tokenizer.pre_tokenizer = Whitespace()

# # Step 2: Wrap it with PreTrainedTokenizerFast
# hf_tokenizer = PreTrainedTokenizerFast(
#     tokenizer_object=tokenizer,
#     unk_token="<unk>",
#     pad_token="<pad>",
#     bos_token="<bos>",
#     eos_token="<eos>"
# )

# # Optional: Save it to disk
# hf_tokenizer.save_pretrained("myarabictokenizer")


In [6]:
from transformers import PreTrainedTokenizerFast

tokenizer = PreTrainedTokenizerFast.from_pretrained("myarabictokenizer")
print(tokenizer.encode("ابتسامه احمر"))  # Example
tokenizer.eos_token_id

[6, 19]


3

In [7]:
# from tokenizers import Tokenizer
# from tokenizers.models import WordLevel
# from tokenizers.pre_tokenizers import Whitespace
# from transformers import PreTrainedTokenizerFast

# # Your vocab_map


# # Step 1: Create a tokenizer using WordLevel
# tokenizer = Tokenizer(WordLevel(vocab=vocab_info["vocab_map"], unk_token="<unk>"))
# tokenizer.pre_tokenizer = Whitespace()

# # Step 2: Wrap it with PreTrainedTokenizerFast
# hf_tokenizer = PreTrainedTokenizerFast(
#     tokenizer_object=tokenizer,
#     unk_token="<unk>",
#     pad_token="<pad>",
#     bos_token="<bos>",
#     eos_token="<eos>"
# )

# # Optional: Save it to disk
# hf_tokenizer.save_pretrained("myarabictokenizer")

In [8]:
device = "cuda" if torch.cuda.is_available() else 'cpu'
device

'cuda'

In [9]:
model = TrOCRMyDecoder(
input_dim                   = 134,
dec_num_layers              = 6,
dec_num_heads               = 8,

d_model                     = 512,
d_ff                        = 1024,

target_vocab_size           = vocab_info['vocab_size'],
eos_token                   = tokenizer.eos_token_id,
sos_token                   = tokenizer.bos_token_id,
pad_token                   = tokenizer.pad_token_id,

enc_dropout                 = 0.2,
dec_dropout                 = 0.2,
pre_train=False,

max_seq_length              = 128 , # Max sequence length for transcripts. Check data verification.
).to(device)

In [10]:
model

TrOCRMyDecoder(
  (encoder): Encoder(
    (embedding): Linear(in_features=134, out_features=512, bias=True)
    (projection): Linear(in_features=512, out_features=512, bias=True)
    (pos_encoding): PositionalEncoding()
    (enc_layers): ModuleList(
      (0-5): 6 x EncoderLayer(
        (mha): MultiHeadAttention(
          (w_qs): Linear(in_features=512, out_features=510, bias=True)
          (w_ks): Linear(in_features=512, out_features=510, bias=True)
          (w_vs): Linear(in_features=512, out_features=510, bias=True)
          (attention): ScaledDotProductAttention(
            (dropout): Dropout(p=0.2, inplace=False)
            (softmax): Softmax(dim=2)
          )
          (fc): Linear(in_features=510, out_features=512, bias=True)
          (dropout): Dropout(p=0.2, inplace=False)
        )
        (ffn): FeedForward(
          (linear_1): Linear(in_features=512, out_features=1024, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (linear_2): Linear(in_f

In [11]:
def num_parameters(mode):
    total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params / 1E6

para = num_parameters(model)
print("#"*10)
print(f"Model Parameters:\n {para}")
print("#"*10)

##########
Model Parameters:
 32.816264
##########


In [ ]:
# from dtrocr.model import AutoSigLMHeadModel
# from dtrocr.config import AutoSigConfig

# config = AutoSigConfig(
#     vocab_size=vocab_info['vocab_size'],
#     input_dim=86,
#     pose_embedding_length=1000,
#     max_position_embeddings=1200,
#     attn_implementation='eager',
#     gpt2_hf_model='aubmindlab/aragpt2-base',
# )

# device = 0 if torch.cuda.is_available() else 'cpu'
# model = AutoSigLMHeadModel(config)
# model

In [13]:
for i, (batch) in enumerate(train_loader):
    inputs = batch["pose_values"]
    targets_shifted= batch["input_ids"][:, :-1]
    targets_golden  = batch["input_ids"][:, 1:]

    inputs          = inputs.to(device)
    targets_shifted = targets_shifted.to(device)
    targets_golden  = targets_golden.to(device)
    break

In [14]:
def train_model(model, train_loader, optimizer):

    model.train()
    batch_bar = tqdm(total=len(train_loader), dynamic_ncols=True, leave=False, position=0, desc="Train")

    total_loss          = 0
    running_loss        = 0.0
    running_perplexity  = 0.0

    for i, (batch) in enumerate(train_loader):
        inputs = batch["pose_values"]
        targets_shifted= batch["input_ids"][:, :-1]
        targets_golden  = batch["input_ids"][:, 1:]
        input_lens = batch["input_lens"]

        inputs          = inputs.to(device)
        targets_shifted = targets_shifted.to(device)
        targets_golden  = targets_golden.to(device)
        optimizer.zero_grad()

        with torch.cuda.amp.autocast():
            # passing the minibatch through the model
            # raw_predictions, attention_weights = model(inputs, inputs_lengths, targets_shifted, targets_lengths)
            raw_predictions, attention_weights = model(inputs, targets_shifted, input_lens)


            padding_mask = torch.logical_not(torch.eq(targets_shifted, tokenizer.pad_token_id))

            # cast the mask to float32
            padding_mask = padding_mask.float()
            loss = loss_func(raw_predictions.transpose(1,2), targets_golden)*padding_mask
            loss = loss.sum() / padding_mask.sum()

        scaler.scale(loss).backward()   # This is a replacement for loss.backward()
        scaler.step(optimizer)          # This is a replacement for optimizer.step()
        scaler.update()                 # This is something added just for FP16

        running_loss        += float(loss.item())
        perplexity          = torch.exp(loss)
        running_perplexity  += perplexity.item()

        # online training monitoring
        batch_bar.set_postfix(
            loss = "{:.04f}".format(float(running_loss / (i + 1))),
            perplexity = "{:.04f}".format(float(running_perplexity / (i + 1)))
        )

        batch_bar.update()

        del inputs, targets_shifted, targets_golden
        torch.cuda.empty_cache()

    running_loss        = float(running_loss / len(train_loader))
    running_perplexity  = float(running_perplexity / len(train_loader))

    batch_bar.close()

    return running_loss, running_perplexity, attention_weights


In [15]:
from sklearn.metrics import f1_score
import jiwer
import Levenshtein


def save_model(model, optimizer, scheduler, metric, epoch, path):
    torch.save(
        {"model_state_dict"         : model.state_dict(),
         "optimizer_state_dict"     : optimizer.state_dict(),
         "scheduler_state_dict"     : scheduler.state_dict() if scheduler is not None else {},
         metric[0]                  : metric[1],
         "epoch"                    : epoch},
         path
    )
def calc_edit_distance(predictions, y, tokenizer, print_example=False):
    dist = 0.0
    cer = 0.0
    wer = 0.0
    char_f1_total = 0.0
    word_f1_total = 0.0

    batch_size, seq_len = predictions.shape

    for batch_idx in range(batch_size):
        # y_sliced    = tokenizer.decode(y[batch_idx, 0 : y_len[batch_idx]], skip_special_tokens=True)
        y_sliced    = tokenizer.decode(y[batch_idx,], skip_special_tokens=True)

        pred_sliced = tokenizer.decode(predictions[batch_idx], skip_special_tokens=True)

        y_string    = "".join(y_sliced)
        pred_string = "".join(pred_sliced)

        cer += jiwer.cer(y_string, pred_string)
        wer += jiwer.wer(y_string, pred_string)
        dist += Levenshtein.distance(pred_string, y_string)

        # Character F1 per sample
        y_chars = list(y_string)
        pred_chars = list(pred_string)

        common_len = min(len(y_chars), len(pred_chars))
        if common_len > 0:
            char_f1 = f1_score(y_chars[:common_len], pred_chars[:common_len], average='micro', zero_division=0)
            char_f1_total += char_f1

        # Word F1 per sample
        y_words = y_string.split()
        pred_words = pred_string.split()
        common_len = min(len(y_words), len(pred_words))
        if common_len > 0:
            word_f1 = f1_score(y_words[:common_len], pred_words[:common_len], average='micro', zero_division=0)
            word_f1_total += word_f1

        if print_example and batch_idx == 0:
            print("\nGround Truth : ", y_string)
            print("Prediction   : ", pred_string)

    dist /= batch_size
    cer /= batch_size
    wer /= batch_size
    char_f1_avg = char_f1_total / batch_size
    word_f1_avg = word_f1_total / batch_size

    return dist, cer, wer, char_f1_avg, word_f1_avg

In [16]:

def validate_fast(model, dataloader):
    model.eval()

    # progress bar
    batch_bar = tqdm(total=len(dataloader), dynamic_ncols=True, leave=False, position=0, desc="Val", ncols=5)

    running_distance = 0.0
    running_cer = 0.0
    running_wer = 0.0
    running_char_f1 = 0.0
    running_word_f1 = 0.0

    for i, (batch) in enumerate(dataloader):
        inputs = batch["pose_values"]
        targets_shifted= batch["input_ids"][:, :-1]
        targets_golden  = batch["input_ids"][:, 1:]
        input_lens = batch["input_lens"]
        inputs          = inputs.to(device)
        targets_shifted = targets_shifted.to(device)
        targets_golden  = targets_golden.to(device)


        with torch.inference_mode():
            greedy_predictions = model.recognize(inputs, input_lens)

        # calculating Levenshtein Distance
        # @NOTE: modify the print_example to print more or less validation examples
        dist, cer, wer, charf1, word_f1 = calc_edit_distance(greedy_predictions, targets_golden, tokenizer, print_example=True)
        running_distance += dist
        running_cer += cer
        running_wer += wer
        running_char_f1 += charf1
        running_word_f1 += word_f1


        # online validation distance monitoring
        batch_bar.set_postfix(
            running_distance = "{:.04f}".format(float(running_distance / (i + 1))),
            running_cer = "{:.04f}".format(float(running_cer / (i + 1))),
            running_wer = "{:.04f}".format(float(running_wer / (i + 1))),
            running_char_f1 = "{:.04f}".format(float(running_char_f1 / (i + 1))),
            running_word_f1 = "{:.04f}".format(float(running_word_f1 / (i + 1)))

        )

        batch_bar.update()

        del inputs, targets_shifted, targets_golden
        torch.cuda.empty_cache()

        if i==4: break      # validating only upon first five batches

    batch_bar.close()
    running_distance /= 5
    running_cer /= 5
    running_wer /= 5
    running_char_f1 /= 5
    running_word_f1 /= 5

    return running_distance, running_cer, running_wer, running_char_f1, running_word_f1


In [17]:
for i, (batch) in enumerate(train_loader):
    inputs = batch["pose_values"]
    targets_shifted= batch["input_ids"][:, :-1]
    targets_golden  = batch["input_ids"][:, 1:]

    inputs          = inputs.to(device)
    targets_shifted = targets_shifted.to(device)
    targets_golden  = targets_golden.to(device)
    print(inputs.shape)
    break

torch.Size([32, 509, 134])


In [18]:
### Copy weights from P1 model to full model
### Freeze the weights of full transformer input embedding, linear projection, and decoderODO make it non-trainable
checkpoint = torch.load("/home/ubuntu/sign/Pose_To_Gloss_Arabic/mlp-decoder-pretrain/checkpoint-best-distance-model.pth")
model.load_state_dict(checkpoint['model_state_dict'])
# for param in model.decoder.parameters():
#     param.requires_grad = False # TODO make it non-trainable
for param in model.encoder.parameters():
    param.requires_grad = False # TODO make it non-trainable

In [19]:
loss_func   = torch.nn.CrossEntropyLoss(ignore_index = tokenizer.pad_token_id)
scaler      = torch.cuda.amp.GradScaler()
optimizer = torch.optim.AdamW(model.parameters(),
                            lr=0.0001,
                            weight_decay=0.01)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,
            T_max = 35, eta_min=1E-8)

/tmp/ipykernel_877/2351929031.py:2: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler      = torch.cuda.amp.GradScaler()


In [20]:
import gc
torch.cuda.empty_cache()
gc.collect()

e                   = 0
best_loss           = 1000
best_distance  = 0
checkpoint_root = os.path.join(os.getcwd(), "mlp-decoder-pretrain-no")
os.makedirs(checkpoint_root, exist_ok=True)

checkpoint_best_loss_model_filename     = 'checkpoint-best-loss-model.pth' 
checkpoint_best_distance_model_filename     = 'checkpoint-best-distance-model.pth'

checkpoint_last_epoch_filename          = 'checkpoint-epoch-'
best_loss_model_path                    = os.path.join(checkpoint_root, checkpoint_best_loss_model_filename)
best_distance_model_path                    = os.path.join(checkpoint_root, checkpoint_best_distance_model_filename)

# epochs = config["epochs"]
epochs = 100

for epoch in range(epochs):

    print("\nEpoch {}/{}".format(epoch+1, epochs))

    curr_lr = float(optimizer.param_groups[0]["lr"])

    train_loss, train_perplexity, attention_weights = train_model(model, train_loader, optimizer)

    print("\nEpoch {}/{}: \nTrain Loss {:.04f}\t Train Perplexity {:.04f}\t Learning Rate {:.04f}".format(
        epoch + 1, epochs, train_loss, train_perplexity, curr_lr))
        
    if (epoch >-1 )and (epoch % 1 == 0):    # validate every 2 epochs to speed up training
        levenshtein_distance, cer, wer, charf1, wordf1 = validate_fast(model, dev_loader)
        print(levenshtein_distance, cer, wer,)
        if best_distance <= levenshtein_distance:
            best_distance = levenshtein_distance
            save_model(model, optimizer, scheduler, ['val_distance', levenshtein_distance], epoch, best_distance_model_path)
            
    # scheduler.step()


Epoch 1/100


Train:   0%|          | 0/297 [00:00<?, ?it/s]

/tmp/ipykernel_877/3904064469.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast():
